# 02 - Model Selection & Evaluation

**Objective:** choose an architecture for the dandelion/grass classifier
under real constraints - small dataset (~400 images), CPU-only
training/serving, fast inference for an interactive API - and define how we
evaluate it.


## 1. Candidate approaches

| Approach | Pros | Cons | Verdict |
|---|---|---|---|
| Custom CNN (from scratch) | Tiny, no download, full control | Needs more data; lower ceiling | **Baseline / offline** |
| **ResNet18 + transfer learning** | Strong ImageNet features -> great on small data; fast to fine-tune | ~45 MB weight download | **Production default** |
| ResNet50 / ViT | Higher ceiling | Overkill, slower on CPU, overfits 400 imgs | Rejected |

Both retained options are exposed via `MODEL_ARCH` (`resnet18` default,
`cnn` baseline) - see `models/model.py`.


In [ ]:
import sys; sys.path.append('..')
import torch
from models.model import build_model

def n_params(m): return sum(p.numel() for p in m.parameters())
cnn = build_model('cnn', num_classes=2, pretrained=False)
resnet = build_model('resnet18', num_classes=2, pretrained=False)
print(f'custom CNN : {n_params(cnn):,} params')
print(f'resnet18   : {n_params(resnet):,} params')


In [ ]:
# Sanity check: both produce a 2-logit output for a 128x128 batch
dummy = torch.randn(2, 3, 128, 128)
print('cnn    ->', tuple(cnn(dummy).shape))
print('resnet ->', tuple(resnet(dummy).shape))


## 2. Why ResNet18 transfer learning is the production default

With only ~400 images, a network trained from scratch tends to overfit. A
ResNet18 pretrained on ImageNet already encodes generic visual features
(edges, textures, colours); we replace only the final layer and fine-tune.
This converges in a few epochs on CPU and generalises better - matching the
documented ~0.90 accuracy / ~0.89 macro-F1.


## 3. Training setup

- **Loss:** cross-entropy - **Optimizer:** Adam (lr=1e-3) - **Scheduler:** StepLR(step=3, gamma=0.1)
- **Epochs:** 5 (default) - **Batch:** 32 - **Split:** 80/20, seed=42
- **Augmentation (train):** resized crop, horizontal flip, colour jitter
- **Normalization:** ImageNet mean/std (required for the pretrained backbone)
- **Tracking:** every run logged to **MLflow** (params, metrics, model, code).


## 4. Evaluation metrics

Reported per epoch and at best checkpoint (`models/train.py::evaluate`):

- **Accuracy** - overall correctness (meaningful here: balanced classes).
- **Macro F1 / precision / recall** - robust if the balance ever shifts.
- **Confusion matrix** - shows *which* class is confused, persisted to
  `metrics_summary.json` for the model card.

Macro-F1 is the headline metric we optimise for and gate releases on.


In [ ]:
# After a training run, read the logged metrics summary (MLflow artifact)
import json, numpy as np
from pathlib import Path
p = Path('../artifacts/metrics_summary.json')
if p.exists():
    s = json.loads(p.read_text())
    print('best acc:', s['best_val_accuracy'], '| F1:', s['val_f1'])
    cm = np.array(s['confusion_matrix'])
    import matplotlib.pyplot as plt
    plt.imshow(cm, cmap='Blues'); plt.title('Confusion matrix')
    plt.xlabel('predicted'); plt.ylabel('true'); plt.colorbar(); plt.show()
else:
    print('Run training first to produce metrics_summary.json')


## 5. Limitations & next steps

- **Data volume** is the main bottleneck - more/diverse images lift the ceiling.
- **Domain shift**: real photos vary by season/lighting -> handled by **drift
  monitoring** + **drift-triggered retraining** (`monitoring/drift`, `retrain/`).
- **Calibration / explainability** (Grad-CAM, probability calibration) are
  natural follow-ups for production trust.

## 6. Reproducibility
Fixed seed (42), MLflow run tracking, and a checkpoint that stores the
architecture + class names so serving rebuilds the exact network.
